Procesado de datos de las baterías 13300 para el TFG de reciclado de baterías de vape.

Primero analizaremos los ciclos y veremos el SoH de las baterías

In [11]:
import os
import shutil
import re
import glob
from pathlib import Path
import numpy as np



def organizar_archivos_por_fecha(carpeta_origen):
    """
    Lee los archivos de una carpeta, extrae la fecha de su nombre 
    y los mueve a subcarpetas organizadas por día.
    """
    
    # Expresión regular para buscar el formato de fecha AAAA_MM_DD
    # \d{4} = 4 dígitos (año), \d{2} = 2 dígitos (mes/día)
    patron_fecha = re.compile(r"(\d{4}_\d{2}_\d{2})")

    # Verificamos que la carpeta de origen exista
    if not os.path.exists(carpeta_origen):
        print(f"Error: La carpeta '{carpeta_origen}' no existe.")
        return

    # Iterar sobre todos los elementos en la carpeta
    for nombre_archivo in os.listdir(carpeta_origen):
        ruta_completa_origen = os.path.join(carpeta_origen, nombre_archivo)

        # Nos aseguramos de procesar solo archivos (ignoramos carpetas)
        if os.path.isfile(ruta_completa_origen):
            
            # Buscamos el patrón de la fecha en el nombre del archivo
            coincidencia = patron_fecha.search(nombre_archivo)
            
            if coincidencia:
                # Extraemos la fecha encontrada (ej. '2026_02_23')
                fecha_carpeta = coincidencia.group(1) 
                
                # Definimos la ruta de la nueva carpeta destino
                ruta_carpeta_destino = os.path.join("data/processed/cicladores", str(i), fecha_carpeta)
                
                # Si la carpeta de esa fecha no existe, la creamos
                if not os.path.exists(ruta_carpeta_destino):
                    os.makedirs(ruta_carpeta_destino)
                
                # Movemos el archivo
                ruta_completa_destino = os.path.join(ruta_carpeta_destino, nombre_archivo)
                shutil.copy(ruta_completa_origen, ruta_completa_destino)
                
                #print(f"Copiado: {nombre_archivo}  ->  Carpeta: {fecha_carpeta}/")
            else:
                print(f"Omitido (sin fecha detectada): {nombre_archivo}")



In [12]:


def trim_txt_files(processed_dir: str = "data/processed/cicladores") -> dict:
    """
    For every .txt file under data/processed/{1-6}/{dates}/:
      - Overwrites each file keeping only the header (line 1) and the last data row.
    For every .dat file in the same folders:
      - Deletes the file entirely.

    Parameters
    ----------
    processed_dir : str
        Path to the 'processed' folder, relative to the notebook location.
        Defaults to "data/processed".

    Returns
    -------
    dict with keys:
        "processed"  – list of .txt files successfully trimmed
        "skipped"    – list of .txt files skipped (< 2 lines or already trimmed)
        "deleted"    – list of .dat files deleted
        "errors"     – list of (filepath, error_message) tuples
    """
    base = Path(processed_dir)
    results = {"processed": [], "skipped": [], "deleted": [], "errors": []}

    def is_valid_folder(p: Path) -> bool:
        return p.parts[-3].isdigit() and 1 <= int(p.parts[-3]) <= 6

    # --- Process .txt files ---
    for filepath in sorted(base.glob("*/*/*.txt")):
        if not is_valid_folder(filepath):
            continue
        try:
            lines = filepath.read_text(encoding="latin-1").splitlines()

            if len(lines) < 2 or len(lines) == 2:
                results["skipped"].append(str(filepath))
                continue

            header   = lines[0]
            last_row = lines[-1]

            filepath.write_text(header + "\n" + last_row + "\n", encoding="latin-1")
            results["processed"].append(str(filepath))

        except Exception as e:
            results["errors"].append((str(filepath), str(e)))

    # --- Delete .dat files ---
    for filepath in sorted(base.glob("*/*/*.dat")):
        if not is_valid_folder(filepath):
            continue
        try:
            filepath.unlink()
            results["deleted"].append(str(filepath))
        except Exception as e:
            results["errors"].append((str(filepath), str(e)))

    # Summary
    print(f"✅ Trimmed : {len(results['processed'])} .txt files")
    print(f"⏭️  Skipped : {len(results['skipped'])} .txt files (already 2 lines or empty)")
    print(f"🗑️  Deleted : {len(results['deleted'])} .dat files")
    print(f"❌ Errors  : {len(results['errors'])} files")
    if results["errors"]:
        for fp, err in results["errors"]:
            print(f"   {fp}: {err}")

    return results

In [13]:
for i in range(1, 7):

    mi_carpeta = f"data/raw/cicladores/ciclador_{i}"
    
    print(f"Procesando: {mi_carpeta} ...")
    
    # Llamamos a la función con la ruta actualizada
    organizar_archivos_por_fecha(mi_carpeta)

print("¡Todas las carpetas han sido organizadas!")

Procesando: data/raw/cicladores/ciclador_1 ...
Procesando: data/raw/cicladores/ciclador_2 ...
Procesando: data/raw/cicladores/ciclador_3 ...
Procesando: data/raw/cicladores/ciclador_4 ...
Procesando: data/raw/cicladores/ciclador_5 ...
Procesando: data/raw/cicladores/ciclador_6 ...
¡Todas las carpetas han sido organizadas!


In [14]:
results = trim_txt_files() #porque ya conoce que tiene que buscar en data/processed

✅ Trimmed : 129 .txt files
⏭️  Skipped : 21 .txt files (already 2 lines or empty)
🗑️  Deleted : 150 .dat files
❌ Errors  : 0 files


Hemos limpiado el sistema de archivos: El resultado en la carpeta processed es una serie de archivos .txt en el que solamente se encuentra el último estado del ciclo que corresponde. Lo que nos interesa a continuación es casar cada archivo txt de Charge con la última carga correspondiente a cada batería.

In [15]:

def build_soh_matrix(baterias_dir: str = "data/processed/baterias") -> np.ndarray:
    """
    Iterates over all battery folders in data/processed/baterias/ and extracts
    the SOH PS (Ah) value from the single data row of each .txt file.

    Expected file structure:
        baterias/{battery_number}/{file}.txt
    Each .txt has exactly 2 lines: a header row and one data row (pre-trimmed).

    Parameters
    ----------
    baterias_dir : str
        Path to the 'baterias' folder, relative to the notebook location.

    Returns
    -------
    np.ndarray
        1 x N matrix (shape (1, N)) with the SOH PS (Ah) value per battery folder,
        sorted by folder name. Folders with missing/invalid files get np.nan.
    """
    base = Path(baterias_dir)

    # Grab all subdirectories, sorted so the order is deterministic
    battery_folders = sorted([p for p in base.iterdir() if p.is_dir()], key=lambda p: int(p.name))
    if not battery_folders:
        print(f"No folders found under '{base}'. Check the path.")
        return np.empty((1, 0))

    soh_values = []

    for folder in battery_folders:
        txt_files = list(folder.glob("*.txt"))

        if not txt_files:
            print(f"⚠️  No .txt file found in '{folder.name}' — inserting NaN.")
            soh_values.append(np.nan)
            continue

        if len(txt_files) > 1:
            print(f"⚠️  Multiple .txt files in '{folder.name}' — using first: {txt_files[0].name}")

        filepath = txt_files[0]

        try:
            lines = filepath.read_text(encoding="latin-1").splitlines()

            # Find the SOH PS (Ah) column index from the header
            header_cols = [col.strip() for col in lines[0].split(",")]
            soh_col_idx = header_cols.index("SOH PS (Ah)")

            # Extract value from the data row
            data_cols = lines[1].split(",")
            soh_value = float(data_cols[soh_col_idx])
            soh_values.append(soh_value)

        except ValueError as e:
            print(f"⚠️  Could not parse '{filepath}': {e} — inserting NaN.")
            soh_values.append(np.nan)
        except Exception as e:
            print(f"❌  Unexpected error with '{filepath}': {e} — inserting NaN.")
            soh_values.append(np.nan)

    print(f"✅ Built SOH matrix of shape {len(soh_values)} from {len(battery_folders)} battery folders.")

    return soh_values

In [19]:
soh_matrix = build_soh_matrix()
# soh_matrix.shape → (1, 36)
# soh_matrix[0, 4] → SOH value of the 5th battery folder

print(soh_matrix)

✅ Built SOH matrix of shape 36 from 36 battery folders.
[0.335735, 0.340149, 0.36167, 0.331654, 0.302161, 0.345978, 0.339422, 0.287277, 0.342146, 0.340374, 0.330512, 0.335763, 0.343665, 0.281199, 0.346292, 0.358724, 0.333173, 0.324965, 0.29439, 0.344466, 0.345323, 0.362929, 0.316819, 0.334309, 0.288062, 0.344176, 0.354084, 0.369454, 0.337994, 0.310809, 0.348332, 0.300527, 0.334904, 0.367375, 0.340035, 0.339868]


In [17]:
def find_best_18(soh_values: np.ndarray) -> np.ndarray:
    """
    Finds the 18 values in soh_values that are closest to each other
    and also as high as possible.

    Strategy:
        1. Use a sliding window of size 18 over the sorted values to find
           the window with the minimum spread (max - min).
        2. Among all windows with the same minimum spread, pick the one
           with the highest mean.

    Parameters
    ----------
    soh_values : np.ndarray
        1D array of SOH PS (Ah) values, one per battery.

    Returns
    -------
    np.ndarray
        18 x 2 matrix where each row is [original_index, soh_value],
        sorted by original index.
    """
    values = np.array(soh_values)
    n = len(values)
    assert n >= 18, f"Need at least 18 values, got {n}."

    # Work on sorted values but keep track of original indices
    sorted_indices = np.argsort(values)
    sorted_values  = values[sorted_indices]

    # Sliding window of size 18: find min spread, then highest mean
    best_window  = 0
    best_spread  = np.inf
    best_mean    = -np.inf

    for i in range(n - 17):
        window_vals = sorted_values[i : i + 18]
        spread = window_vals[-1] - window_vals[0]  # max - min within window
        mean   = window_vals.mean()

        if spread < best_spread or (spread == best_spread and mean > best_mean):
            best_spread = spread
            best_mean   = mean
            best_window = i

    # Retrieve original indices for the best window
    selected_sorted_indices = sorted_indices[best_window : best_window + 18]

    # Build the 18x2 result matrix: [original_index, soh_value]
    result = np.array([[idx + 1, values[idx]] for idx in sorted(selected_sorted_indices)])

    print(f"✅ Selected 18 batteries | spread: {best_spread:.6f} | mean SOH: {best_mean:.6f}")
    return result

In [18]:
result = find_best_18(soh_matrix)

print(result)

✅ Selected 18 batteries | spread: 0.013119 | mean SOH: 0.340210
[[ 1.        0.335735]
 [ 2.        0.340149]
 [ 6.        0.345978]
 [ 7.        0.339422]
 [ 9.        0.342146]
 [10.        0.340374]
 [12.        0.335763]
 [13.        0.343665]
 [15.        0.346292]
 [17.        0.333173]
 [20.        0.344466]
 [21.        0.345323]
 [24.        0.334309]
 [26.        0.344176]
 [29.        0.337994]
 [33.        0.334904]
 [35.        0.340035]
 [36.        0.339868]]


Con esto, ya hemos encontrado las 18 baterías más compatibles para formar un sólo pack. Ahora, solo queda montarlo y ensayarlo.